Splitsen van datasets vóór nb08. 
Dit is een fundamenteel principe van temporele validatie: alles wat in nb08 wordt gefitst (encoders, scalers, binning-grenzen, target encoding) mag alleen op de trainingsset worden bepaald. Als je dat onderscheid pas in nb09 maakt, riskeer je subtiele leakage.

In [ ]:
# ── nb07c — Train/Holdout split ───────────────────────────────────────
import pandas as pd
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from mechelen_parking import get_default_paths, get_quality_mask

PATHS = get_default_paths()
DATA_PROC = PATHS.data_processed

mad_st = pd.read_parquet(DATA_PROC / "MAD_shortterm.parquet")
mad_clean = mad_st[get_quality_mask(mad_st)].copy()

train = mad_clean[mad_clean["year"].isin([2020, 2023, 2024])].copy()
holdout = mad_clean[mad_clean["year"] == 2025].copy()

print(f"Train  : {len(train):>7,} rijen | jaren: {sorted(train['year'].unique())}")
print(f"Holdout: {len(holdout):>7,} rijen | jaren: {sorted(holdout['year'].unique())}")
print(f"Overlap check (moet 0 zijn): {len(set(train.index) & set(holdout.index))}")

train.to_parquet(DATA_PROC / "train.parquet", index=False)
holdout.to_parquet(DATA_PROC / "holdout.parquet", index=False)

print("
✅ Opgeslagen:")
print(f"   data_processed/train.parquet   — {len(train):,} rijen")
print(f"   data_processed/holdout.parquet — {len(holdout):,} rijen")


Train  : 129,837 rijen | jaren: [np.int32(2020), np.int32(2023), np.int32(2024)]
Holdout:  87,600 rijen | jaren: [np.int32(2025)]
Overlap check (moet 0 zijn): 0

✅ Opgeslagen:
   data_processed/train.parquet   — 129,837 rijen
   data_processed/holdout.parquet — 87,600 rijen


#### Datastructuur na nb07c
```text
data_processed/
├── MAD_shortterm.parquet    ← origineel, nooit aanpassen
├── MAD_longterm.parquet     ← origineel, nooit aanraken*
├── train.parquet            ← 2020 + 2023 + 2024, gefilterd
└── holdout.parquet          ← 2025, gefilterd, nooit fitten op

```

* LT-data: correct, je doet er niets mee in de pipeline. De enige uitzondering is de statische high_lt_pressure-dummy (P Hoogstraat en P Komet = 1), maar die wordt hard-coded afgeleid uit de parkeernaam — je hoeft MAD_longterm.parquet daar niet voor te lezen.